In [211]:
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_percentage_error


# For 2D analysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
# from utils import period2freq, freq2period

# For PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import time
import glob
import os

# Datasize in KB
data_size_kb = {'4mb': 4096, '16mb': 16384, '64mb': 65536,
            '256mb': 262144, '512mb': 524288, '1gb': 1048576,
            '5gb': 5242880, '50gb': 52428800, '100gb': 104857600,
            '300gb': 314572800,}

# Key Parameters
IOR_PARAMS = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'trMiB', "storageType"]

TARGET_PARAMS = [ "bestStorage" ]
op_dict = {0: "write", 1: "read"}


In [212]:
# My utility functions
import utils.perf_visualize as pv

# Parameter Notes for Datalife:
Each entry in the table represent only one single edge in the workflow. An directed edge connects a **fileName** and a **taskName**, representing data access.

---
- **operation**: The type of I/O operation {0: "write", 1: "read"}, value 1 represents read (e.g. a directed edge edge from a **fileName** to a **taskName**), value 0 represents write (e.g. a directed edge edge from a **taskName** to a **fileName**)
- **randomOffset**: The type of data access pattern { 0: "sequential file access", 1: "random file access"}
- **transferSize**: Average I/O size of the particular I/O operation to a file, calculated from aggregateFilesizeMB/opCount
- **aggregateFilesizeMB**: Total I/O size of a particular I/O operation to a file for a task
- **numTasks**: Number of parallel tasks for this particular task
- **totalTime**: The total I/O time of of a particular I/O operation to a file for a task
- **numNodes**: Number of nodes used for this particular task
- **tasksPerNode**: numTasks/numNodes for a task
- **bwMiB**: transferRate of a particular I/O operation to a file for a task, calculated from aggregateFilesizeMB/totalTime
- **storageType**: The storage type used in this task. {0: "localssd", 1: "beegfs/pfs", 2: "lustre", 3: "unknown"}
- **opCount**: the number of I/O operation count of a particular I/O operation to a file for a task
- **taskName**: the task name that is running for a particular workflow
- **taskPID**: the task PID
- **fileName**: the name of file that a I/O operation is for

In [213]:
def transform_store_code(storage_type):
    if storage_type == "localssd":
        store_code = 0
    elif storage_type == "beegfs" or storage_type == "pfs":
        store_code = 1
    elif storage_type == "lustre":
        store_code = 2
    else:
        store_code = 3
    return store_code

def decode_store_code(store_code):
    if store_code == 0:
        storage_type = "localssd"
    elif store_code == 1:
        storage_type = "beegfs"
    elif store_code == 2:
        storage_type = "lustre"
    else:
        storage_type = "unknown"
    return storage_type



In [214]:
def merge_by_test_param(ssd_df, beegfs_df):
    # Compare and find the row with only storage_type and trMiB columns different but others are the same
    # Merge the dataframes on columns other than 'storageType' and 'trMiB'
    merge_columns = ['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB', 'numTasks', 'numNodes', 'tasksPerNode', 'opCount'] # 'totalTime', 
    merged_df = pd.merge(ssd_df, beegfs_df, on=merge_columns, suffixes=('_ssd', '_beegfs'), how='outer')

    # Remove columns storageType_ssd and storageType_beegfs
    merged_df.drop(['storageType_ssd', 'storageType_beegfs'], axis=1, inplace=True)
    
    # # Check if cloumens 'operation_ssd' and 'operation_beegfs' has the same values, if yes combine to 1 column 'operation'
    # merged_df['operation'] = merged_df.apply(lambda row: 0 if row['operation_ssd'] == row['operation_beegfs'] else -1, axis=1)

    # Compare 'trMiB' values and add 'selectStorage' column
    merged_df['selectStorage'] = merged_df.apply(lambda row: 0 if row['trMiB_ssd'] > row['trMiB_beegfs'] else 1, axis=1)

    # Debug: Print unique values of 'operation' column before and after merge
    print("Unique 'operation' values in ssd_df:", ssd_df['operation'].unique())
    print("Unique 'operation' values in beegfs_df:", beegfs_df['operation'].unique())
    print("Unique 'operation' values in merged_df:", merged_df['operation'].unique())
    
    print(merged_df.head(5))
    # print shaoe
    print(f"merged_df.shape: {merged_df.shape}")
    print(f"ssd_df.shape: {ssd_df.shape}")
    print(f"beegfs_df.shape: {beegfs_df.shape}")

    return merged_df




In [215]:
def is_sequential(numbers):
    if not numbers:  # Check if the list is empty
        return False

    sorted_numbers = sorted(numbers)  # Sort the numbers
    return all(sorted_numbers[i] + 1 == sorted_numbers[i + 1] for i in range(len(sorted_numbers) - 1))


def get_stat_file_pids(all_files):
    # Extract target tasks from blk_files
    target_tasks = set()
    for blk_file in all_files:
        # Get the filename without the path
        filename = os.path.basename(blk_file)
        # repalce ".local" for now
        filename = filename.replace(".local", "")
        
        # Split filename by '.'
        parts = filename.split('.')
        # print(f"get_stat_file_pids() : parts = {parts}")
        if len(parts) >= 3:
            # Get the target task from the -3 extension
            task = parts[-3]
            target_tasks.add(task)
    target_tasks = sorted(target_tasks)
    return target_tasks

def add_stat_to_df(trial_folder, numTasksWrite, numNodes, 
                   monitor_timer_stat_io, operation, fname, task_name, task_pid, store_code):

    fname = fname.replace(".local", ".")
    # fileName removed the last 4 extensions
    fileName = ".".join(fname.split(".")[:-4])
    # fileName keep only the basename
    fileName = os.path.basename(fileName)
                        
    # Get write statistics
    # print(f"monitor_timer_stat_write = {monitor_timer_stat_write}")
    tmp_write_stat = {}
    tmp_write_stat['aggregateFilesizeMB'] = pv.file_size_to_mb(monitor_timer_stat_io[2])

    tmp_write_stat['numTasks'] = numTasksWrite
    tmp_write_stat['numNodes'] = numNodes
    tmp_write_stat['tasksPerNode'] = tmp_write_stat['numTasks']/tmp_write_stat['numNodes']
    tmp_write_stat['transferSize'] = monitor_timer_stat_io[2]/monitor_timer_stat_io[1]
    tmp_write_stat['operation'] = int(operation)
    tmp_write_stat['totalTime'] = monitor_timer_stat_io[0]
    tmp_write_stat['trMiB'] = pv.file_size_to_mb(monitor_timer_stat_io[2]/monitor_timer_stat_io[0])
    tmp_write_stat['storageType'] = store_code
    tmp_write_stat['opCount'] = monitor_timer_stat_io[1]
    tmp_write_stat['taskName'] = task_name
    tmp_write_stat['taskPID'] = task_pid
    tmp_write_stat['fileName'] = fileName
    # if w_fname == "":
    #     print(f"Write file not found for task_pid[{task_pid}] but has write stat [{tmp_write_stat}]")
    
    op = "w"
    if operation == 1: op = "r"

    # find the w_blk_trace_jsons files with the current task_pid
    w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{task_pid}.{op}_blk_trace.json")
    write_pattern = 0 # 0: seq, 1: rand        
    for w_blk_trace_json in w_blk_trace_jsons:
        with open(w_blk_trace_json) as f:
            w_blk_trace_data = json.load(f)
            # print(w_blk_trace_data)
            blk_list = w_blk_trace_data['io_blk_range']
            if blk_list[2] == -2:
                # FIXME: For now only single read and write
                write_pattern = 1
                break
    tmp_write_stat['randomOffset'] = write_pattern
    
    return tmp_write_stat
    


# TODO: find task PID's input and output to match script name
def get_wf_result_df(tests, wf_params, target_tasks,
                     numTasksWrite=1, numTasksRead=1, numNodes=1, storageType="localssd"):

    wf_df = pd.DataFrame(columns=wf_params)

    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders} ")

    store_code = transform_store_code(storageType)
    
    for trial_folder in wf_trial_folders:
        # Find all json files in the trial folder
        # wf_json_files = glob.glob(f"{trial_folder}/*.json")


        datalife_jsons = glob.glob(f"{trial_folder}/*.datalife.json")
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        
        # print(f"blk_files = {blk_files}")
        # print(f"datalife_jsons = {datalife_jsons}")


        # Convert target_tasks to a sorted list if needed
        target_tasks = get_stat_file_pids(blk_files)
        
        # print(f"target_tasks = {target_tasks}")
        
        for datalife_json in datalife_jsons:
            task_pid = datalife_json.split("/")[-1].split(".")[1]
            # print(f"task_pid = {task_pid}")
            if task_pid in target_tasks:
                # print(f"Found taks {task_pid} in target_tasks")
                monitor_timer_stat = {}
                with open(datalife_json) as f:
                    # Get task pid from the filename monitor_timer.pid.datalife.json
                    try:
                        datalife_data = json.load(f)
                    except:
                        print(f"Error loading empty file [{f}]")

                    # Get the first key
                    task_name = list(datalife_data.keys())[0]
                    # Select 'monitor', not 'system' and 'local'
                    monitor_timer_stat = datalife_data[task_name]['monitor']
                    system_timer_stat = datalife_data[task_name]['system']

                # print(monitor_timer_stat)
 
                monitor_timer_targets = ["read", "write"] # , "close" , "seek", "stat"
                
                blk_trace_fname = [f for f in blk_files if f".{task_pid}." in f]
                # print(f"blk_trace_fname = {blk_trace_fname}")
                for fname in blk_trace_fname:
                    # Check if the file starts with r_ or w_
                    if ".r_blk_trace." in fname:
                        # Collect all statistics for read
                        read_time = 0
                        for k, v_list in monitor_timer_stat.items():
                            # read_time+= v_list[0]
                            if k in monitor_timer_targets:
                                read_time += v_list[0]
                        # Get read statistics
                        monitor_timer_stat_read = monitor_timer_stat['read']
                        # Update read time
                        monitor_timer_stat_read[0] = read_time
                        if pv.file_size_to_mb(monitor_timer_stat_read[2]) == 0:
                            print(f"No read stat for task_name[{task_name}] task_pid[{task_pid}]")
                        else:
                        
                            tmp_read_stat = add_stat_to_df(trial_folder, numTasksRead, numNodes,
                                            monitor_timer_stat_read, 1, fname, task_name, task_pid, store_code)
                            wf_df = wf_df._append(tmp_read_stat, ignore_index=True)
                        
                    elif ".w_blk_trace." in fname:
                        # Collect all statistics for read
                        write_time = 0
                        for k, v_list in monitor_timer_stat.items():
                            # write_time+= v_list[0]
                            if k in monitor_timer_targets:
                                write_time += v_list[0]
                        # Get write statistics                        
                        monitor_timer_stat_write = monitor_timer_stat['write'] # list with [time_sec, op_cnt, io_size]
                        # Update write time
                        monitor_timer_stat_write[0] = write_time
                        if pv.file_size_to_mb(monitor_timer_stat_write[2]) == 0:
                            print(f"No write stat for task_name[{task_name}] task_pid[{task_pid}]")
                        else:
                            monitor_timer_stat_write = monitor_timer_stat['write'] # list with [time_sec, op_cnt, io_size]
                            tmp_write_stat = add_stat_to_df(trial_folder, numTasksWrite, numNodes, 
                                        monitor_timer_stat_write, 0, fname, task_name, task_pid, store_code)
                            wf_df = wf_df._append(tmp_write_stat, ignore_index=True)
                            
    return wf_df

In [216]:
# Load 1kgenome data
onekg_data_path = "./1kgenome_data/fastflow_tests"


# Key Parameters
wf_params = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'trMiB', 'storageType',
            'opCount','taskName','taskPID', 'fileName']
target_tasks = ["python"] # omit srun from 1kgenome run

all_wf_df = pd.DataFrame(columns=wf_params)

def get_test_folder_dfs(test_folder, wf_params, target_tasks, 
                        storageType="localssd", numNodes=1):
    folder_dfs = pd.DataFrame(columns=wf_params)

    for tests in test_folder:
        # check of test folder starts with seq or par
        if tests.startswith("seq"):
            numTasksWrite = 1
            numTasksRead = 1
        else:
            # get the number after _ps
            num_tasks = int(tests.split("_")[-1].split("ps")[1])
            numTasksWrite = num_tasks
            numTasksRead = num_tasks

        # io_size_dfs
        wf_df = get_wf_result_df(f"{onekg_data_path}/{tests}", wf_params, target_tasks, 
                                numTasksWrite=numTasksWrite, numTasksRead=numTasksRead, 
                                storageType=storageType,numNodes=numNodes)
        print(wf_df.head(5))
        # Print size of df
        print(f"df shape: {wf_df.shape}")

        # corr_matrix(wf_df, storageType)
        folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
    return folder_dfs

wf_pfs_df = pd.DataFrame(columns=wf_params)
# test_folders = ['par_3000_1n_pfs_ps300', 'par_6000_1n_pfs_ps300', 
#                 'par_9000_1n_pfs_ps300'] par_3000_10n_shm_ps300

test_folders = ['par_3000_10n_nfs_ps300']

wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders, 
                                        wf_params, target_tasks, numNodes=10,
                                        storageType="pfs"), ignore_index=True)



Trial folders: ['./1kgenome_data/fastflow_tests/par_3000_10n_nfs_ps300/300_p_10n_NFS_t1'] 


/tmp/ipykernel_879/3299527481.py:152: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_read_stat, ignore_index=True)


  operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0         1            0   12532.331361             2.019848      300   
1         0            0  117100.750000             0.893408      300   
2         1            0   12532.331361             2.019848      300   
3         1            0   12532.331361             2.019848      300   
4         1            0    8191.853947          1355.538332      300   

   totalTime numNodes  tasksPerNode        trMiB storageType opCount taskName  \
0   0.001043       10          30.0  1936.320800           1     169   python   
1   0.001506       10          30.0   593.188958           1       8   python   
2   0.002549       10          30.0   792.331175           1     169   python   
3   0.004055       10          30.0   498.069065           1     169   python   
4  10.759385       10          30.0   125.986602           1  173512   python   

       taskPID              fileName  
0  50951-dc111  sifted.SIFT.chr2.tx

/tmp/ipykernel_879/1825593022.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
/tmp/ipykernel_879/1825593022.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders,


In [217]:
print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)

  operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0         1            0   12532.331361             2.019848      300   
1         0            0  117100.750000             0.893408      300   
2         1            0   12532.331361             2.019848      300   
3         1            0   12532.331361             2.019848      300   
4         1            0    8191.853947          1355.538332      300   

   totalTime numNodes  tasksPerNode        trMiB storageType opCount taskName  \
0   0.001043       10          30.0  1936.320800           1     169   python   
1   0.001506       10          30.0   593.188958           1       8   python   
2   0.002549       10          30.0   792.331175           1     169   python   
3   0.004055       10          30.0   498.069065           1     169   python   
4  10.759385       10          30.0   125.986602           1  173512   python   

       taskPID              fileName  
0  50951-dc111  sifted.SIFT.chr2.tx

In [218]:
def match_script_name(tests):

    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders} ")

    pid_input_output_dict = {}
    
    for trial_folder in wf_trial_folders:
        # Find all json files in the trial folder
        # wf_json_files = glob.glob(f"{trial_folder}/*.json")

        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        unique_pids = get_stat_file_pids(blk_files)

        # monitor_tasks = get_stat_file_pids(datalife_jsons)
        # # Find the subset tasks pids from both target_tasks and monitor_tasks
        # common_tasks = set(target_tasks).intersection(monitor_tasks)
        # print(f"Target tasks num {len(target_tasks)}")
        # print(f"Common tasks: {common_tasks}")

        for pid in unique_pids:
            pid_input_output_dict[pid] = {"input": [], 
                                           "output": [],
                                           "prevTask": "",
                                           "taskName": ""}
            # Find the blk_trace_jsons files with the current task_pid # FIXME: .local should be removed later
            w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.w_blk_trace.json")
            r_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.r_blk_trace.json")
            
            # FIXME: current replace .local with empty string
            w_blk_trace_jsons = [f.replace(".local", "") for f in w_blk_trace_jsons]
            r_blk_trace_jsons = [f.replace(".local", "") for f in r_blk_trace_jsons]

            if w_blk_trace_jsons:
                # Extract the file path and file name
                w_file_path = w_blk_trace_jsons[0]
                w_file_name_parts = w_file_path.split(".")
                # Remove the last 3 extensions
                w_file_name = '.'.join(w_file_name_parts[:-3])
                w_file_path_modified = '/'.join(w_file_path.split("/")[:-1]) + '/' + w_file_name
                # print(f"w_file_path_modified = {w_file_path_modified}")
                w_file_basename = w_file_path_modified.split("/")[-1]
                pid_input_output_dict[pid]['output'].append(w_file_basename)

            if r_blk_trace_jsons:
                # Extract the file path and file name
                r_file_path = r_blk_trace_jsons[0]
                r_file_name_parts = r_file_path.split(".")
                # Remove the last 3 extensions
                r_file_name = '.'.join(r_file_name_parts[:-3])
                # Reconstruct the path with the modified file name
                r_file_path_modified = '/'.join(r_file_path.split("/")[:-1]) + '/' + r_file_name
                # print(f"r_file_path_modified = {r_file_path_modified}")
                r_file_basename = r_file_path_modified.split("/")[-1]
                pid_input_output_dict[pid]['input'].append(r_file_basename)

        # print(pid_input_output_dict)
        return pid_input_output_dict

def get_wf_pid_script_dict(test_folder):

    all_wf_dict= {}

    for tests in test_folder:
        # check of test folder starts with seq or par
        if tests.startswith("seq"):
            numTasksWrite = 1
            numTasksRead = 1
        else:
            # get the number after _ps
            num_tasks = int(tests.split("_")[-1].split("ps")[1])
            numTasksWrite = num_tasks
            numTasksRead = num_tasks

        # io_size_dfs
        wf_dict = match_script_name(f"{onekg_data_path}/{tests}")

        # # corr_matrix(wf_df, storageType)
        all_wf_dict.update(wf_dict)
    return all_wf_dict


all_wf_dict = get_wf_pid_script_dict(test_folders)

print(all_wf_dict)


Trial folders: ['./1kgenome_data/fastflow_tests/par_3000_10n_nfs_ps300/300_p_10n_NFS_t1'] 
{'10532-dc155': {'input': ['sifted.SIFT.chr6.txt'], 'output': ['chr6-GBR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10542-dc155': {'input': ['columns.txt'], 'output': ['chr6-EUR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10546-dc155': {'input': ['sifted.SIFT.chr6.txt'], 'output': ['chr6-AMR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10551-dc155': {'input': ['chr6n.tar.gz'], 'output': ['chr6-ALL.tar.gz'], 'prevTask': '', 'taskName': ''}, '10578-dc155': {'input': ['columns.txt'], 'output': ['chr6-SAS.tar.gz'], 'prevTask': '', 'taskName': ''}, '10581-dc155': {'input': ['columns.txt'], 'output': ['chr6-EAS.tar.gz'], 'prevTask': '', 'taskName': ''}, '10585-dc155': {'input': ['columns.txt'], 'output': ['chr6-AFR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10600-dc155': {'input': ['sifted.SIFT.chr6.txt'], 'output': ['chr6-GBR-freq.tar.gz'], 'prevTask': '', 'taskName': ''}, '10601-dc155': {'input': ['

In [219]:
# Add prevTask column
wf_pfs_df['prevTask'] = ""
print(wf_pfs_df.head(5))

  operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0         1            0   12532.331361             2.019848      300   
1         0            0  117100.750000             0.893408      300   
2         1            0   12532.331361             2.019848      300   
3         1            0   12532.331361             2.019848      300   
4         1            0    8191.853947          1355.538332      300   

   totalTime numNodes  tasksPerNode        trMiB storageType opCount taskName  \
0   0.001043       10          30.0  1936.320800           1     169   python   
1   0.001506       10          30.0   593.188958           1       8   python   
2   0.002549       10          30.0   792.331175           1     169   python   
3   0.004055       10          30.0   498.069065           1     169   python   
4  10.759385       10          30.0   125.986602           1  173512   python   

       taskPID              fileName prevTask  
0  50951-dc111  sifted.SIF

In [220]:
import re


# Function to match a file path with patterns in task definitions
def matches_pattern(file_path, patterns):
    # Convert patterns from the JSON file to regex and match against file_path
    file_name = os.path.basename(file_path)
    # print(f"matches_pattern(): file_name = {file_name}")
    # file_name = "columns.txt"
    # patterns = ["columns\\.txt"]
    for pattern in patterns:
        try:
            # Compile the pattern to regex
            regex_pattern = re.compile(pattern)
            # print(f"Trying regex pattern: {regex_pattern.pattern}")
            if regex_pattern.fullmatch(file_name):
                # print(f"Match found: {file_name} matches {regex_pattern.pattern}")
                return True
        except re.error as e:
            print(f"Invalid regex pattern: {pattern}. Error: {e}")
            
    # print(f"No match found for file_name = {file_name}")
    return False
        
def assign_task_names(tasks, task_definitions):
    for task_pid, details in tasks.items():
        input_paths = details['input']
        output_paths = details['output']
        task_name = 'unknown'

        # Iterate through each task definition to find matches
        for task, definition in task_definitions.items():
            # Check if any output path matches the output patterns (unique to each task)
            output_patterns = definition['outputs']
            output_match = any(matches_pattern(op, output_patterns) for op in output_paths)
            if output_match:
                # Assign the task name based on the output match
                if task_name == 'unknown':
                    task_name = task
                # Now check input patterns to determine the prevTask
                for prevTask, v in definition['predecessors'].items():
                    input_patterns = v['inputs']  # a list of input patterns
                    input_match = any(matches_pattern(ip, input_patterns) for ip in input_paths)

                    if input_match and tasks[task_pid]['prevTask'] == '':
                        # print(f"-- Assining prevTask: {prevTask} for task_pid: {task_pid} with input: {input_paths} matching pattern: {input_patterns}")
                        # If an input match is found, update the prevTask
                        tasks[task_pid]['prevTask'] = prevTask
                        break  # Stop once a match is found for prevTask

                break
        tasks[task_pid]['taskName'] = task_name

# print(len(all_wf_dict))
print(f"all_wf_dict = {all_wf_dict}")

# Load task ordering json file
task_order_dict = {}
with open(f"{onekg_data_path}/1kg_script_order.json") as f:
    task_order_dict = json.load(f)

print(task_order_dict)

# Fill in task names
assign_task_names(all_wf_dict, task_order_dict)
# Unique list of taskNames
taskNames = set([v['taskName'] for v in all_wf_dict.values()])
print(f"Unique taskNames: {taskNames}")
print(all_wf_dict)

# wf_pfs_df =  assign_task_names(wf_pfs_df, task_order_dict)
# print(wf_pfs_df.head(5))

all_wf_dict = {'10532-dc155': {'input': ['sifted.SIFT.chr6.txt'], 'output': ['chr6-GBR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10542-dc155': {'input': ['columns.txt'], 'output': ['chr6-EUR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10546-dc155': {'input': ['sifted.SIFT.chr6.txt'], 'output': ['chr6-AMR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10551-dc155': {'input': ['chr6n.tar.gz'], 'output': ['chr6-ALL.tar.gz'], 'prevTask': '', 'taskName': ''}, '10578-dc155': {'input': ['columns.txt'], 'output': ['chr6-SAS.tar.gz'], 'prevTask': '', 'taskName': ''}, '10581-dc155': {'input': ['columns.txt'], 'output': ['chr6-EAS.tar.gz'], 'prevTask': '', 'taskName': ''}, '10585-dc155': {'input': ['columns.txt'], 'output': ['chr6-AFR.tar.gz'], 'prevTask': '', 'taskName': ''}, '10600-dc155': {'input': ['sifted.SIFT.chr6.txt'], 'output': ['chr6-GBR-freq.tar.gz'], 'prevTask': '', 'taskName': ''}, '10601-dc155': {'input': ['chr6n.tar.gz'], 'output': ['chr6-EAS-freq.tar.gz'], 'prevTask': '', 'taskName

In [221]:
# Create a mapping from taskPID to taskName
task_pid_to_name = {pid: info['taskName'] for pid, info in all_wf_dict.items()}
print(task_pid_to_name)
# Update the DataFrame with the taskName
wf_pfs_df['taskName'] = wf_pfs_df['taskPID'].map(task_pid_to_name).fillna('unknown')

# Create a mapping from taskPID to prevTask
task_pid_to_prev_task = {pid: info['prevTask'] for pid, info in all_wf_dict.items()}
print(task_pid_to_prev_task)
# add prevTask column to the DataFrame
wf_pfs_df['prevTask'] = wf_pfs_df['taskPID'].map(task_pid_to_prev_task).fillna('unknown')

# FIXME: temporary 1k genome data fix
# remove dataframe rows with all_wf_df['taskName'] == 'unknown'
wf_pfs_df = wf_pfs_df[wf_pfs_df['taskName'] != 'unknown']
# remove rows with filename contaiing string "SIFT.chr*.vcf"
for chrom in range(0, 11):
    wf_pfs_df = wf_pfs_df[~wf_pfs_df['fileName'].str.contains(f"SIFT.chr{chrom}.vcf")]
    
    
# Adjust dataframe prevTask
for index, row in wf_pfs_df.iterrows():
    if row['operation'] == 0:
        if row['taskName'] == '':
            # Update taskName for write tasks to "" (empty string)
            wf_pfs_df.at[index, 'taskName'] = 'none'
    else:
        # Adjust read task predecessors
        taskName = row['taskName']
        fileName = row['fileName']
        task_definition = task_order_dict[taskName]
        for task, inputs in task_definition['predecessors'].items():
            input_patterns = inputs['inputs']
            if matches_pattern(fileName, input_patterns):
                wf_pfs_df.at[index, 'prevTask'] = task

# Print the updated DataFrame
print(wf_pfs_df.head(5))

{'10532-dc155': 'mutation_overlap', '10542-dc155': 'mutation_overlap', '10546-dc155': 'mutation_overlap', '10551-dc155': 'mutation_overlap', '10578-dc155': 'mutation_overlap', '10581-dc155': 'mutation_overlap', '10585-dc155': 'mutation_overlap', '10600-dc155': 'frequency', '10601-dc155': 'frequency', '10603-dc155': 'frequency', '10605-dc155': 'frequency', '10607-dc155': 'frequency', '10609-dc155': 'frequency', '10611-dc155': 'frequency', '107356-dc273': 'individuals', '107361-dc273': 'individuals', '107389-dc273': 'individuals', '107401-dc273': 'individuals', '107412-dc273': 'individuals', '107425-dc273': 'individuals', '107441-dc273': 'individuals', '107464-dc273': 'individuals', '107499-dc273': 'individuals', '107503-dc273': 'individuals', '107505-dc273': 'individuals', '107507-dc273': 'individuals', '107509-dc273': 'individuals', '107511-dc273': 'individuals', '107514-dc273': 'individuals', '107516-dc273': 'individuals', '107518-dc273': 'individuals', '107520-dc273': 'individuals', 

In [222]:
# Modify numTasks by mapping the "parallelism" from task_order_dict based on taskName
# Create a mapping from taskName to parallelism
task_name_to_parallelism = {task: info['parallelism'] for task, info in task_order_dict.items()}
print(task_name_to_parallelism)
wf_pfs_df['numTasks'] = wf_pfs_df['taskName'].map(task_name_to_parallelism).fillna(1)
# Update the tasksPerNode column
wf_pfs_df['tasksPerNode'] = wf_pfs_df['numTasks'] / wf_pfs_df['numNodes']



{'individuals': 300, 'individuals_merge': 10, 'sifting': 10, 'mutation_overlap': 10, 'frequency': 10}


In [223]:
# Save the updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}.csv', index=False)


In [224]:
# Calculate I/O time per taskName
task_io_time_total = wf_pfs_df.groupby('taskName')['totalTime'].sum()

task_io_time_adjust = {}
total_wf_io_time = 0
for task, io_time in task_io_time_total.items():
    # Adjust I/O time by parallelism
    io_time_adjusted = io_time / task_name_to_parallelism[task]
    task_io_time_adjust[task] = io_time_adjusted
    total_wf_io_time+=io_time_adjusted
    
    print(f" {task}: {io_time_adjusted} (sec)")

# print(task_io_time_adjust)
print(f"Total I/O time per workflow: {total_wf_io_time}")

 frequency: 1.543192525 (sec)
 individuals: 85.21312523882001 (sec)
 individuals_merge: 2.1070231398 (sec)
 mutation_overlap: 0.1277093464 (sec)
 sifting: 29.739298406 (sec)
Total I/O time per workflow: 118.73034865602001


In [225]:
# Read from file "./master_ior_df.csv"
df_ior = pd.read_csv("./master_ior_df.csv")
print(df_ior.shape)

# oscache size is 25GiB
oscacheSizeMB = 25 * 1024  # Convert to MiB

(4788, 30)


In [226]:
# Function to calculate transferRate based on bounds
def calculate_transferRate_storage(df_ior, tasks_per_node, transfer_size, transferRate_column):
    # Sort df_ior by tasksPerNode and transferSize
    df_ior_sorted = df_ior.sort_values(by=['tasksPerNode', 'transferSize'])
    
    # Find lower and upper bounds for tasksPerNode
    lower_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] <= tasks_per_node]
    upper_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] >= tasks_per_node]
    
    # Take lowest/highest task df if no lower/higher bound found
    if lower_tasks.empty:
        lower_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] == df_ior_sorted['tasksPerNode'].min()]
    if upper_tasks.empty:
        upper_tasks = df_ior_sorted[df_ior_sorted['tasksPerNode'] == df_ior_sorted['tasksPerNode'].max()]
    
    low_bound_tasks = lower_tasks.iloc[-1]
    high_bound_tasks = upper_tasks.iloc[0]
    
    # Filter df_ior by tasksPerNode bounds
    df_ior_within_bounds = df_ior_sorted[
        (df_ior_sorted['tasksPerNode'] >= low_bound_tasks['tasksPerNode']) &
        (df_ior_sorted['tasksPerNode'] <= high_bound_tasks['tasksPerNode'])
    ]
    
    if df_ior_within_bounds.empty:
        raise ValueError("No rows found within tasksPerNode bounds.")
    
    # Sort within bounds by transferSize
    df_ior_within_bounds = df_ior_within_bounds.sort_values(by='transferSize')
    
    # Check if there are enough rows to find transferSize bounds
    low_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] < transfer_size]
    high_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] > transfer_size]
    
    # Take lowest/highest transferSize in df if no lower/higher bound found
    if low_bound_transfer.empty:
        low_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] == df_ior_within_bounds['transferSize'].min()]
    if high_bound_transfer.empty:
        high_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] == df_ior_within_bounds['transferSize'].max()]
    
    low_bound = low_bound_transfer.iloc[-1]
    high_bound = high_bound_transfer.iloc[0]
    
    # Perform linear interpolation to estimate the transferRate
    low_size = low_bound['transferSize']
    high_size = high_bound['transferSize']
    low_tr = low_bound[transferRate_column]
    high_tr = high_bound[transferRate_column]
    
    if (high_size - low_size) == 0:
        estimated_tr = low_tr
    else:
        estimated_tr = low_tr + (transfer_size - low_size) * (high_tr - low_tr) / (high_size - low_size)
    return float(estimated_tr)


# Iterate through each row of scaled_wf_df
for index, row in wf_pfs_df.iterrows():
    # Find matching rows in df_ior based on operation
    conditions = (
        (df_ior['operation'] == row['operation'])
    ) 
    df_ior_matched = df_ior[conditions]

    if not df_ior_matched.empty:
        # Find bounds for the transferSize
        if len(df_ior_matched) > 1:
            try:
                aggregateFilesizeMB = row['aggregateFilesizeMB']
                
                # Calculate estimated transferRate
                estimated_trMiB_ssd = calculate_transferRate_storage(df_ior_matched, row['tasksPerNode'], row['transferSize'],'trMiB_ave_ssd')
                wf_pfs_df.at[index, 'estimated_trMiB_ssd'] = estimated_trMiB_ssd
                estimated_trMiB_beegfs = calculate_transferRate_storage(df_ior_matched, row['tasksPerNode'], row['transferSize'],'trMiB_ave_beegfs')
                wf_pfs_df.at[index, 'estimated_trMiB_beegfs'] = estimated_trMiB_beegfs

            except ValueError as e:
                print(f"Row {index}: Error calculating transferRate - {e}")
                wf_pfs_df.at[index, 'estimated_tr'] = 0
        else:
            print(f"Row {index}: Not enough data to determine bounds.")
            wf_pfs_df.at[index, 'estimated_tr'] = 0
    else:
        print(f"Row {index}: No matching rows in df_ior.")
        wf_pfs_df.at[index, 'estimated_tr'] = 0

    
# Save updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}_tr_estimated.csv', index=False)

# Split dataframe to different task names and save
unique_task_names = wf_pfs_df['taskName'].unique()

for task_name in unique_task_names:
    task_df = wf_pfs_df[wf_pfs_df['taskName'] == task_name].copy()
    
    write_df = task_df[task_df['operation'] == 0]
    read_df = task_df[task_df['operation'] == 1]
    for storage in ['ssd', 'beegfs']:
        e_write_var = write_df[f'estimated_trMiB_{storage}'].var()
        e_read_var = read_df[f'estimated_trMiB_{storage}'].var()
        # Update the DataFrame with the calculated variance according to operation
        task_df.loc[task_df['operation'] == 0, f'var_estimated_trMiB_{storage}'] = e_write_var
        task_df.loc[task_df['operation'] == 1, f'var_estimated_trMiB_{storage}'] = e_read_var
    a_write_var = write_df[f'trMiB'].var()
    a_read_var = read_df[f'trMiB'].var()
    # Update the DataFrame with the calculated variance according to operation
    task_df.loc[task_df['operation'] == 0, 'var_trMiB'] = a_write_var
    task_df.loc[task_df['operation'] == 1, 'var_trMiB'] = a_read_var
    # # Calculated the estimated_trMiB variance
    # for storage in ['ssd', 'beegfs']:
    #     task_df[f'estimated_trMiB_{storage}_var'] = task_df[f'estimated_trMiB_{storage}'].var()
    task_df.to_csv(f'{test_folders[0]}_{task_name}_tr_estimated.csv', index=False)

- PT: the preceeding task within the workflow
# Constructs SPM values in dataframe
- ETR^{store}_{group}(IOT,Ave,D, n):  calculate the estimated transfer rate for a storage for a task (taskPID) per IOType, Average I/O size, Data size, and tasks parallelism
- SPM: Stage performance matching of producer-consumer
    - Construct a list of producer consumer tables
    - Per table, we calculate ht SPM number

In [227]:
# Construct the SPM Here:
pc_df_list = []

task_name_list = list(wf_pfs_df['taskName'].unique())
print(f"Unique task names: {task_name_list}")
# task_name_list.append('none') # for first stage in the workflow

print(wf_pfs_df.columns)



Unique task names: ['mutation_overlap', 'sifting', 'individuals', 'frequency', 'individuals_merge']
Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB',
       'numTasks', 'totalTime', 'numNodes', 'tasksPerNode', 'trMiB',
       'storageType', 'opCount', 'taskName', 'taskPID', 'fileName', 'prevTask',
       'estimated_trMiB_ssd', 'estimated_trMiB_beegfs'],
      dtype='object')


In [ ]:
def create_empty_df(other_df, taskName, prevTask="none", operation=0):
    MAXTR = 250000  # Maximum transfer rate in MiB/s
    other_df_len = len(other_df)
    
    # Dictionary to hold columns for DataFrame
    df_dict = {
        'operation': [operation] * other_df_len,
        'randomOffset': [0] * other_df_len,
        'transferSize': [1] * other_df_len,
        'aggregateFilesizeMB': [1] * other_df_len,
        'numTasks': [1] * other_df_len,
        'totalTime': [0] * other_df_len,
        'numNodes': [1] * other_df_len,
        'tasksPerNode': [1] * other_df_len,
        'trMiB': [MAXTR] * other_df_len,
        'storageType': ["n/a"] * other_df_len,
        'opCount': [1] * other_df_len,
        'taskName': [taskName] * other_df_len,
        'taskPID': ["none"] * other_df_len,
        'fileName': ["none"] * other_df_len,
        'prevTask': [prevTask] * other_df_len,
        'estimated_trMiB_ssd': [MAXTR] * other_df_len,
        'estimated_trMiB_beegfs': [MAXTR] * other_df_len
    }
    
    # Convert dict to DataFrame
    empty_df = pd.DataFrame(df_dict)
    
    # Debugging output
    print(f"Constructed empty DataFrame:\n{empty_df.head(5)}")
    
    return empty_df

# TODO: Construct Wrokflow Graph from the DataFrame
# Iterate through each row to add the nodes and edges


In [228]:



for taskName in task_name_list:    
    # Filter the DataFrame by taskName and make it a new DataFrame
    producer_df = wf_pfs_df[(wf_pfs_df['taskName'] == taskName) & (wf_pfs_df['operation'] == 0)].copy()
    consumer_df = wf_pfs_df[(wf_pfs_df['prevTask'] == taskName) & (wf_pfs_df['operation'] == 1)].copy()
    
    # TODO: need to use the same PID
    # if both empty, raise error
    if producer_df.empty and consumer_df.empty:
        print(f"No producer and consumer dataframes found for taskName: {taskName}")
        raise ValueError(f"No producer and consumer dataframes found for taskName: {taskName}")
    
    # Case when taskName is the first task in the workflow, no producer_df
    if producer_df.empty:
        fileName_list = consumer_df['fileName'].unique()
        print(f"No producer dataframe found for taskName: {taskName} for files: {fileName_list}")
        # Create an empty producer_df
        producer_df = create_empty_df(taskName, consumer_df, prevTask="none", operation=0)
    
    if consumer_df.empty:
        fileName_list = producer_df['fileName'].unique()
        print(f"No consumer dataframe found for taskName: {taskName} for files: {fileName_list}")
        # Create an empty consumer_df
        consumer_df = create_empty_df("none", producer_df, prevTask=taskName, operation=1)

    # check producer-consumer pair
    consumer_task_list = consumer_df[['taskName']].unique()
    print(f"The current producer consumer pair is {taskName}:{consumer_task_list}\n-------------------")
    
    # Calculate the SPM for the producer and consumer dataframes
    pc_df = pd.DataFrame(columns=wf_params)
    for storage_type in ["ssd", "beegfs"]:
        # Calculate rate of producer and consume
        producer_df[f'io_intensity_{storage_type}'] = producer_df['opCount'] * producer_df['aggregateFilesizeMB'] / producer_df[f'estimated_trMiB_{storage_type}']
        consumer_df[f'io_intensity_{storage_type}'] = consumer_df['opCount'] * consumer_df['aggregateFilesizeMB'] /consumer_df[f'estimated_trMiB_{storage_type}']
        # Create empty column for SPM
        consumer_df[f'spm_{storage_type}'] = 0
        producer_df[f'spm_{storage_type}'] = 0
    
        # TODO: loop through consumer to calculate the spm_{storage_type} value
        for index, row in consumer_df.iterrows():
            # fileName = row['fileName']
            # producer_row = producer_df[producer_df['fileName'] == fileName]
            taskPID = row['taskPID']
            producer_row = producer_df[producer_df['taskPID'] == taskPID]
            
            if not producer_row.empty:
                # Calculate the SPM and save to consumer_df
                consumer_df.at[index, f'spm_{storage_type}'] = producer_row[f'io_intensity_{storage_type}'] / row[f'io_intensity_{storage_type}']
                # # Check value at consumer_df
                # print(f"SPM_{storage_type} for fileName: {fileName} is {consumer_df.at[index, f'spm_{storage_type}']}")
            else:
                # Print a warning
                print(f"WARNING: Producer row not found for fileName: {fileName} for taskName: {taskName}")
        
        # Check updated column
        print(f"Updated consumer_df with SPM_{storage_type} on taskName: {taskName}")
        
    # Concatenate the producer and consumer dataframes
    pc_df = pd.concat([producer_df, consumer_df], ignore_index=True)
    # Check the updated pc_df spm columns
    print(f"Updated pc_df with SPM columns")
    print(pc_df.head(5))

    # TODO
    # Do storage selection based on estimated_trMiB_{storage_type} and spm_{storage_type}
    # Compare between the two storage types and select the higher SPM save to new column name
    # If 

    select_storage_list = []
    # Iterate through each row of pc_df
    for index, row in pc_df.iterrows():
        # if spm is 0, select the storage with higher estimated_trMiB
        base_storage = 'beegfs'
        comp_storage = 'ssd'
        if row[f'spm_{base_storage}'] == 0 and row[f'spm_{comp_storage}'] == 0:
            # USe "n/a" for write operation
            if row['operation'] == 0:
                select_storage_list.append("n/a")
            else:
                if row[f'estimated_trMiB_{base_storage}'] > row[f'estimated_trMiB_{comp_storage}']:
                    select_storage_list.append(base_storage)
                else:
                    select_storage_list.append(comp_storage)
        else:
            # select based of spm
            if row[f'spm_{base_storage}'] > row[f'spm_{comp_storage}']:
                select_storage_list.append(base_storage)
            else:
                select_storage_list.append(comp_storage)

    # Add the select_storage_list to the pc_df
    pc_df['select_storage'] = select_storage_list

    # Save the pc_df to csv
    file_name = f'{test_folders[0]}_{taskName}_spm.csv'
    pc_df.to_csv(file_name, index=False)
    print(f"Saved SPM to {file_name}")
 


No consumer dataframe found for taskName: mutation_overlap for files: ['chr2-GBR.tar.gz' 'chr3-SAS.tar.gz' 'chr2-ALL.tar.gz' 'chr10-EAS.tar.gz'
 'chr7-ALL.tar.gz' 'chr10-AMR.tar.gz' 'chr4-AFR.tar.gz' 'chr5-SAS.tar.gz'
 'chr4-SAS.tar.gz' 'chr9-AMR.tar.gz' 'chr1-ALL.tar.gz' 'chr10-GBR.tar.gz'
 'chr3-ALL.tar.gz' 'chr6-GBR.tar.gz' 'chr6-ALL.tar.gz' 'chr8-AFR.tar.gz'
 'chr7-EUR.tar.gz' 'chr1-SAS.tar.gz' 'chr5-GBR.tar.gz' 'chr5-ALL.tar.gz'
 'chr10-AFR.tar.gz' 'chr7-SAS.tar.gz' 'chr8-EAS.tar.gz' 'chr6-SAS.tar.gz'
 'chr4-AMR.tar.gz' 'chr1-AMR.tar.gz' 'chr8-AMR.tar.gz' 'chr1-EUR.tar.gz'
 'chr3-AFR.tar.gz' 'chr2-EAS.tar.gz' 'chr9-EAS.tar.gz' 'chr6-EUR.tar.gz'
 'chr8-EUR.tar.gz' 'chr2-EUR.tar.gz' 'chr3-EAS.tar.gz' 'chr8-ALL.tar.gz'
 'chr10-SAS.tar.gz' 'chr5-EAS.tar.gz' 'chr1-AFR.tar.gz' 'chr5-AMR.tar.gz'
 'chr5-EUR.tar.gz' 'chr2-SAS.tar.gz' 'chr7-AMR.tar.gz' 'chr10-ALL.tar.gz'
 'chr9-EUR.tar.gz' 'chr2-AFR.tar.gz' 'chr4-ALL.tar.gz' 'chr4-EAS.tar.gz'
 'chr1-GBR.tar.gz' 'chr10-EUR.tar.gz' 'chr6-AFR.

AttributeError: 'DataFrame' object has no attribute 'unique'

In [199]:
print(pc_df.shape)

(150, 22)


In [197]:
# Split dataframe to different task names and save
unique_task_names = pc_df['taskName'].unique()
print(f"Unique task names: {unique_task_names}")

for task_name in unique_task_names:
    # Filter the DataFrame by taskName and make it a new DataFrame
    task_df = pc_df[pc_df['taskName'] == task_name].copy()
    
    # # Calculated the estimated_trMiB variance
    # for storage in ['ssd', 'beegfs']:
    #     task_df[f'estimated_trMiB_{storage}_var'] = task_df[f'estimated_trMiB_{storage}'].var()
    task_df.to_csv(f'{test_folders[0]}_{task_name}_spm.csv', index=False)
    # Calculate the average of spm
    spm_avg_ssd = task_df['spm_ssd'].mean()
    spm_avg_beegfs = task_df['spm_beegfs'].mean()
    print(f"Average SPM for {task_name}: SSD {spm_avg_ssd}, BeeGFS {spm_avg_beegfs}")

Unique task names: ['individuals_merge' 'mutation_overlap' 'frequency']
Average SPM for individuals_merge: SSD 0.0, BeeGFS 0.0
Average SPM for mutation_overlap: SSD 0.0, BeeGFS 0.0
Average SPM for frequency: SSD 0.0, BeeGFS 0.0
